# 04 — Frozen MiniFASNetV2 evaluation on the Kaggle CASIA-FASD image copy

**Run after Notebook 02 and Notebook 03 are frozen.** Mount `minhnh2107/casiafasd`, the final 100K seed-100 outputs of Notebook 02 and 03, and the exact SCRFD-500M `det_500m.onnx`. Edit only the mount paths below. Run all cells, then download the output ZIP.

This Kaggle copy contains **JPEG frames**, not video files. The explorer found 2,408 color test frames grouped by filename into 360 source-video IDs and 30 subjects (IDs 1–30), plus a paired depth branch that this RGB model does not use. Its `train_img` and `test_img` directories overlap for subjects 1–20 (240 video IDs, 1,403 identical frame identifiers). At the user's direction, this notebook evaluates **all `test_img/color`** because the four models were trained only on CelebA-Spoof. It reports this copy's test subset, **not** the original CASIA-FASD official subject-disjoint benchmark. No CASIA data trains, tunes, calibrates, or selects a model.

The protocol below is fixed before predictions: up to 16 uniformly spaced *available extracted frames* per source video, no replacement on detection failure, at least 4 valid crops per video, shared SCRFD detections and 2.7x BGR crops, mean frame logit score per video, each model's own frozen CelebA-Val threshold. The primary unit is the source-video ID reconstructed from filenames. Subject-level bootstrap describes uncertainty across these 30 subjects, not training-seed uncertainty. The JPEG sample does not support claims about unprovided frames or the full original video.


In [7]:
# Kaggle setup. If packages are already available, this cell makes no changes.
import importlib.util, subprocess, sys
missing = [name for name in ('insightface', 'onnx', 'onnxruntime', 'tabulate') if importlib.util.find_spec(name) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])


In [8]:
import os

target_path = "/kaggle/input/datasets"

if not os.path.exists(target_path):
    # Đề phòng trường hợp dùng đường dẫn tương đối từ thư mục làm việc hiện tại
    alt_path = "kaggle/input/datasets"
    target_path = alt_path if os.path.exists(alt_path) else target_path

if not os.path.exists(target_path):
    print(f"❌ Không tìm thấy thư mục: {target_path}")
    print("Danh mục hiện có trong /kaggle/input:", os.listdir("/kaggle/input") if os.path.exists("/kaggle/input") else "Thư mục /kaggle/input không tồn tại")
else:
    print(f"🔍 Cấu trúc: {target_path}\n" + "="*40)
    for root, dirs, files in os.walk(target_path):
        depth = root[len(target_path):].count(os.sep)
        if depth >= 3:
            dirs.clear()
            continue
            
        indent = "    " * depth
        folder_name = os.path.basename(root) or target_path
        print(f"{indent}📂 {folder_name}/")
        
        sub_indent = "    " * (depth + 1)
        for i, file in enumerate(files):
            if i >= 5:
                print(f"{sub_indent}... ({len(files) - 5} files khác)")
                break
            print(f"{sub_indent}📄 {file}")

🔍 Cấu trúc: /kaggle/input/datasets
📂 datasets/
    📂 minhnh2107/
        📂 casiafasd/
    📂 takasugi1303/
        📂 minifasnet-100k-scratch/
        📂 scrfd-det/
            📄 det_500m.onnx
        📂 minifasnet-100k-pretrained/


In [9]:
from pathlib import Path
from collections import Counter
import hashlib, json, math, re, shutil, zipfile
import cv2, numpy as np, pandas as pd, onnx, onnxruntime as ort
from insightface import model_zoo
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt
from IPython.display import display, FileLink

CASIA_ROOT = "/kaggle/input/datasets/minhnh2107/casiafasd"
SCRATCH_RESULT_MOUNT = "/kaggle/input/datasets/takasugi1303/minifasnet-100k-scratch/minifasnet_final100k_random_seed100_v2"
PRETRAINED_RESULT_MOUNT = "/kaggle/input/datasets/takasugi1303/minifasnet-100k-pretrained/minifasnet_final100k_pretrained_seed100_v2"
SCRFD_MODEL_PATH = "/kaggle/input/datasets/takasugi1303/scrfd-det/det_500m.onnx"
OUTPUT_ROOT = Path('/kaggle/working/minifasnetv2_casia_fasd_frozen_eval')
FRAMES_PER_VIDEO = 16
MIN_VALID_FRAMES = 4
BOOTSTRAP_SEED = 4242
N_BOOTSTRAP = 2000
SCRFD_SHA256 = '5e4447f50245bbd7966bd6c0fa52938c61474a04ec7def48753668a9d8b4ea3a'
SOURCES = {'scratch': ('minifasnet_final100k_random_seed100_v2', '02_minifasnetv2_final100k_randominit_vanilla_vs_csmr_v2.ipynb', SCRATCH_RESULT_MOUNT),
           'pretrained': ('minifasnet_final100k_pretrained_seed100_v2', '03_minifasnetv2_final100k_pretrained_vanilla_vs_csmr_v2.ipynb', PRETRAINED_RESULT_MOUNT)}
MODEL_IDS = ('scratch_vanilla', 'scratch_csmr', 'pretrained_vanilla', 'pretrained_csmr')
EXPECTED_CODES = {'1': (0,'real','low'), '2': (0,'real','normal'), 'HR_1': (0,'real','high'),
                  '3': (1,'warped','low'), '4': (1,'warped','normal'), 'HR_2': (1,'warped','high'),
                  '5': (1,'cut','low'), '6': (1,'cut','normal'), 'HR_3': (1,'cut','high'),
                  '7': (1,'replay','low'), '8': (1,'replay','normal'), 'HR_4': (1,'replay','high')}
FILENAME_RE = re.compile(r'^(?P<subject>\d+)_(?P<code>HR_[1-4]|[1-8])\.avi_(?P<frame>\d+)_(?P<label>real|fake)\.jpg$', re.I)

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()

def save_csv(frame, filename):
    frame.to_csv(OUTPUT_ROOT/filename, index=False)

def json_read(path):
    return json.loads(Path(path).read_text(), parse_constant=lambda x: (_ for _ in ()).throw(ValueError(x)))

def resolve_casia():
    if CASIA_ROOT is not None:
        p = Path(CASIA_ROOT)
        if not p.is_dir(): raise FileNotFoundError(p)
        return p.resolve()
    candidates = [Path('/kaggle/input/datasets/minhnh2107/casiafasd'), Path('/kaggle/input/casiafasd')]
    base = Path('/kaggle/input')
    if base.is_dir(): candidates += [p for p in base.iterdir() if p.is_dir() and 'casia' in p.name.lower() and 'fasd' in p.name.lower()]
    found = sorted({p.resolve() for p in candidates if p.is_dir()}, key=str)
    if len(found) != 1: raise RuntimeError(f'Ambiguous CASIA mount: {found}; set CASIA_ROOT')
    return found[0]

def resolve_detector():
    if SCRFD_MODEL_PATH is not None:
        paths = [Path(SCRFD_MODEL_PATH)]
    else:
        paths = sorted(Path('/kaggle/input').rglob('det_500m.onnx'))
    if len(paths) != 1: raise RuntimeError(f'Expected exactly one det_500m.onnx: {paths}; set SCRFD_MODEL_PATH')
    if sha256(paths[0]) != SCRFD_SHA256: raise RuntimeError('SCRFD model SHA256 differs from Notebook 00 frozen detector')
    return paths[0]

def resolve_results(mount, expected):
    if '<' in mount or '>' in mount: raise ValueError(f'Edit Kaggle model mount: {mount}')
    mount = Path(mount)
    if not mount.is_dir(): raise FileNotFoundError(mount)
    if any(x in mount.as_posix().lower() for x in ('20k', 'pilot', 'tuning')): raise RuntimeError('Rejected nonfinal model mount')
    # Match the complete final output directory name, never a prefix.
    dirs = [p for p in [mount, *mount.rglob(expected)] if p.is_dir() and p.name == expected]
    zips = [p for p in mount.rglob(expected+'.zip') if p.is_file()]
    if len(dirs)+len(zips) != 1: raise RuntimeError(f'Expected one {expected} directory or ZIP under {mount}; got dirs={dirs}, zips={zips}')
    if dirs: return dirs[0]
    archive = zips[0]
    with zipfile.ZipFile(archive) as z:
        names = z.namelist()
        if not names or any(n.startswith('/') or '..' in Path(n).parts for n in names): raise RuntimeError('Unsafe model ZIP paths')
        if any(not n.startswith(expected+'/') for n in names): raise RuntimeError('Model ZIP has unexpected root')
        target = OUTPUT_ROOT/'extracted'/expected
        if not target.exists(): z.extractall(OUTPUT_ROOT/'extracted')
    return target

def validate_model(root, regime, branch, source_notebook, source_output):
    folder = root/branch
    paths = {name: folder/name for name in ('best.onnx','runtime_config.json','onnx_validation.json')}
    if not all(p.is_file() for p in paths.values()): raise FileNotFoundError(f'Missing final artifacts: {paths}')
    cfg, validation = json_read(paths['runtime_config.json']), json_read(paths['onnx_validation.json'])
    training = json_read(folder/'config.json')
    expected_init = 'random' if regime == 'scratch' else 'pretrained'
    required_training = {'schema':'final_mini_pair_v1','initialization':expected_init,
                         'train_seed':100,'branch':branch,'train_manifest':'final_train100k',
                         'val_manifest':'final_val15k'}
    for key, value in required_training.items():
        if training.get(key) != value: raise RuntimeError(f'{regime}_{branch}: training {key} mismatch')
    frozen = json_read(root/'comparison/frozen_development.json')
    model_hash = sha256(paths['best.onnx'])
    if model_hash != cfg.get('model_sha256') or model_hash != validation.get('onnx_sha256'): raise RuntimeError('ONNX hash mismatch: '+regime+'_'+branch)
    if cfg.get('training_checkpoint_sha256') != frozen[branch]['sha']: raise RuntimeError('Checkpoint freeze mismatch')
    if not math.isclose(float(cfg['calibrated_logit_threshold']), float(frozen[branch]['locked']), abs_tol=1e-12): raise RuntimeError('Val threshold freeze mismatch')
    contract = {'model_file':'best.onnx','input_name':'input','output_name':'logits',
                'input_shape':['batch',3,80,80], 'output_shape':['batch',3],
                'class_names':['real','physical_spoof','digital_spoof'],
                'color_order':'BGR','input_range':'[0,1]', 'mean':None,'std':None,
                'apply_gamma':False,'bbox_expansion_factor':2.7,
                'crop_mode':'minifasnet_train_v1',
                'threshold_score_definition':'logit_real - logsumexp(logit_physical, logit_digital)',
                'threshold_source':'CelebA Val only'}
    for key, value in contract.items():
        if cfg.get(key) != value: raise RuntimeError(f'{regime}_{branch}: {key} mismatch: {cfg.get(key)}')
    graph = onnx.load(str(paths['best.onnx'])); onnx.checker.check_model(graph)
    if len(graph.graph.input)!=1 or len(graph.graph.output)!=1: raise RuntimeError('ONNX graph IO count mismatch')
    if graph.graph.input[0].name!='input' or graph.graph.output[0].name!='logits': raise RuntimeError('ONNX IO names mismatch')
    in_dims = graph.graph.input[0].type.tensor_type.shape.dim
    out_dims = graph.graph.output[0].type.tensor_type.shape.dim
    if [d.dim_value for d in in_dims[1:]] != [3,80,80] or [d.dim_value for d in out_dims[1:]] != [3]:
        raise RuntimeError('ONNX graph shape mismatch')
    blocked = ('fft','dft','spectral','complex')
    if any(any(x in node.op_type.lower() for x in blocked) for node in graph.graph.node): raise RuntimeError('Spectral inference op found')
    return {'model_id':regime+'_'+branch,'initialization_regime':regime,'method':branch,
            'source_notebook':source_notebook,'source_output_name':source_output,
            'onnx_path':str(paths['best.onnx']),'onnx_sha256':model_hash,
            'training_checkpoint_sha256':cfg['training_checkpoint_sha256'],
            'locked_celeba_val_logit_threshold':float(cfg['calibrated_logit_threshold']),
            'input_resolution':'80x80','color_order':'BGR','input_range':'[0,1]',
            'bbox_expansion_factor':2.7,'threshold_source':'CelebA Val only'}

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CASIA = resolve_casia()
detector_path = resolve_detector()
provenance = []
for regime, (name, notebook, mount) in SOURCES.items():
    root = resolve_results(mount, name)
    for branch in ('vanilla','csmr'):
        provenance.append(validate_model(root, regime, branch, notebook, name))
assert tuple(x['model_id'] for x in provenance) == MODEL_IDS
save_csv(pd.DataFrame(provenance), 'model_provenance.csv')
print('CASIA:', CASIA, 'SCRFD:', detector_path)
display(pd.DataFrame(provenance).drop(columns='onnx_path'))


CASIA: /kaggle/input/datasets/minhnh2107/casiafasd SCRFD: /kaggle/input/datasets/takasugi1303/scrfd-det/det_500m.onnx


,model_id,initialization_regime,method,source_notebook,source_output_name,onnx_sha256,training_checkpoint_sha256,locked_celeba_val_logit_threshold,input_resolution,color_order,input_range,bbox_expansion_factor,threshold_source
0,scratch_vanilla,scratch,vanilla,02_minifasnetv2_final100k_randominit_vanilla_v...,minifasnet_final100k_random_seed100_v2,e6fd9473932f8e63fbe43b295c440b96b80da06208d4ec...,7d562b8a74fae0e0d336cd625b8c61f96f04a5b68fa4e1...,-0.383404,80x80,BGR,"[0,1]",2.7,CelebA Val only
1,scratch_csmr,scratch,csmr,02_minifasnetv2_final100k_randominit_vanilla_v...,minifasnet_final100k_random_seed100_v2,68979db20e194cb61bb1c9c28716a882015ba645ebd107...,096041f0548d6195ee159c863eb93d85f715398dfd173d...,0.156794,80x80,BGR,"[0,1]",2.7,CelebA Val only
2,pretrained_vanilla,pretrained,vanilla,03_minifasnetv2_final100k_pretrained_vanilla_v...,minifasnet_final100k_pretrained_seed100_v2,1ebe008f795215d0deb38de9275c7f8399283a9541f2bc...,e91120a89db7a2594f418bf261e6e6923fa0b63eeb7af2...,-0.905557,80x80,BGR,"[0,1]",2.7,CelebA Val only
3,pretrained_csmr,pretrained,csmr,03_minifasnetv2_final100k_pretrained_vanilla_v...,minifasnet_final100k_pretrained_seed100_v2,34f018d87aa2bc73d87bfe4a342e7cafb8933ae51fa35c...,ced6af1a626deb750617ff29e07889f3d05f958452e251...,-8.833043,80x80,BGR,"[0,1]",2.7,CelebA Val only


In [10]:
# Rebuild the video/frame manifests from this exact Kaggle image-copy layout.
test_color = CASIA/'test_img'/'test_img'/'color'
train_color = CASIA/'train_img'/'train_img'/'color'
if not test_color.is_dir() or not train_color.is_dir(): raise RuntimeError('Unexpected CASIA image-copy layout; do not guess split')
def read_split(folder):
    output = []
    for path in sorted(folder.rglob('*')):
        if not path.is_file(): continue
        if path.parent != folder: raise RuntimeError('Unexpected nested media path: '+str(path))
        match = FILENAME_RE.fullmatch(path.name)
        if not match: raise RuntimeError('Unknown CASIA filename: '+str(path))
        g = match.groupdict(); sid = int(g['subject']); code = g['code'].upper() if g['code'].upper().startswith('HR_') else g['code']
        label, attack, quality = EXPECTED_CODES[code]
        if g['label'].lower() != ('real' if label == 0 else 'fake'): raise RuntimeError('Filename label/code conflict: '+str(path))
        output.append({'subject_id':sid,'video_code':code,'video_id':f'{sid}_{code}.avi',
                       'frame_index':int(g['frame']),'image_path':str(path),
                       'binary_label':label,'attack_type':attack,'quality':quality})
    frame = pd.DataFrame(output)
    if frame.empty or frame.image_path.duplicated().any() or frame.duplicated(['video_id','frame_index']).any(): raise RuntimeError('Empty/duplicate CASIA manifest')
    return frame.sort_values(['subject_id','video_id','frame_index']).reset_index(drop=True)

all_test = read_split(test_color)
train_names = read_split(train_color)
videos = all_test.groupby('video_id',sort=True).agg(subject_id=('subject_id','first'),video_code=('video_code','first'),
    binary_label=('binary_label','first'),attack_type=('attack_type','first'),quality=('quality','first'),
    available_frames=('image_path','size')).reset_index()
for vid, group in all_test.groupby('video_id'):
    if len(group[['subject_id','video_code','binary_label','attack_type','quality']].drop_duplicates()) != 1: raise RuntimeError('Inconsistent source-video metadata: '+vid)
if set(all_test.subject_id) != set(range(1,31)) or len(videos)!=360 or (videos.available_frames < 1).any(): raise RuntimeError('Unexpected test subject/video inventory')
if int((videos.binary_label==0).sum())!=90 or int((videos.binary_label==1).sum())!=270: raise RuntimeError('Unexpected real/attack video counts')
if any(set(g.video_code)!=set(EXPECTED_CODES) for _,g in videos.groupby('subject_id')): raise RuntimeError('Missing or duplicate video code per subject')
shared_subjects = sorted(set(all_test.subject_id)&set(train_names.subject_id))
shared_videos = sorted(set(all_test.video_id)&set(train_names.video_id))
shared_frame_ids = set(zip(all_test.video_id,all_test.frame_index)) & set(zip(train_names.video_id,train_names.frame_index))
# This overlap is explicitly retained, never described as an official subject-disjoint test.
assert len(shared_subjects)==20 and len(shared_videos)==240 and len(shared_frame_ids)==1403
selected = []
for _, group in all_test.groupby('video_id',sort=True):
    group = group.sort_values('frame_index')
    positions = np.unique(np.linspace(0,len(group)-1,min(FRAMES_PER_VIDEO,len(group)),dtype=int))
    selected.append(group.iloc[positions])
frame_manifest = pd.concat(selected,ignore_index=True).sort_values(['subject_id','video_id','frame_index']).reset_index(drop=True)
assert not frame_manifest.duplicated(['video_id','frame_index']).any()
videos['planned_frames'] = videos.video_id.map(frame_manifest.groupby('video_id').size())
save_csv(videos,'casia_test_video_manifest.csv')
save_csv(frame_manifest,'casia_frame_manifest.csv')
protocol = {'dataset':'CASIA-FASD Kaggle image copy minhnh2107/casiafasd',
 'dataset_role':'untouched external confirmation for these four CelebA-trained models',
 'test_split':'all test_img/test_img/color; nonofficial subject-overlapping Kaggle image split',
 'expected_test_subjects':'image-copy IDs 1..30', 'test_videos':360,
 'test_color_frames_available':len(all_test),'test_color_frames_planned':len(frame_manifest),
 'train_test_overlap':{'subjects':len(shared_subjects),'video_ids':len(shared_videos),'frame_ids':len(shared_frame_ids)},
 'model_sources':[x[0] for x in SOURCES.values()], 'model_ids':MODEL_IDS,
 'frames_per_video':FRAMES_PER_VIDEO,'min_valid_frames':MIN_VALID_FRAMES,
 'frame_sampling':'uniform linspace over available extracted JPEG frames, fixed before detection',
 'detector':'SCRFD-500M det_500m.onnx','detector_sha256':SCRFD_SHA256,
 'detector_threshold':0.5,'detector_input_sizes':[[640,640],[480,480],[320,320]],
 'face_selection':'largest SCRFD box, confidence tie break; no source bbox annotation',
 'crop':'MiniFASNet minifasnet_train_v1 2.7x','resolution':'80x80','color_order':'BGR','input_range':'[0,1]',
 'score':'logit_real - logsumexp(logit_physical, logit_digital)',
 'video_aggregation':'mean valid-frame score', 'threshold_source':'each model own CelebA Val only',
 'target_calibration':False,'target_training':False,'target_adaptation':False,
 'primary_unit':'reconstructed source video','bootstrap':'paired subject bootstrap',
 'bootstrap_seed':BOOTSTRAP_SEED,'bootstrap_replicates':N_BOOTSTRAP,
 'casia_root':str(CASIA),'source_jpeg_limit':'Original full videos are unavailable in this copy'}
(OUTPUT_ROOT/'evaluation_protocol.json').write_text(json.dumps(protocol,indent=2,allow_nan=False)+'\n')
print('Frozen protocol saved before detection/inference. Planned frames:',len(frame_manifest))
print('Train/test copy overlap:',protocol['train_test_overlap'])


Frozen protocol saved before detection/inference. Planned frames: 2405
Train/test copy overlap: {'subjects': 20, 'video_ids': 240, 'frame_ids': 1403}


In [11]:
# Shared deterministic SCRFD localization and exact MiniFASNet training crop.
providers = ['CUDAExecutionProvider','CPUExecutionProvider'] if 'CUDAExecutionProvider' in ort.get_available_providers() else ['CPUExecutionProvider']
detector = model_zoo.get_model(str(detector_path),providers=providers)
if detector is None: raise RuntimeError('Could not load exact SCRFD-500M detector')
detector.prepare(ctx_id=0 if providers[0]=='CUDAExecutionProvider' else -1,input_size=(640,640),det_thresh=0.50)

def detect(image, size):
    try: return detector.detect(image,input_size=size,max_num=0,metric='default')[0]
    except TypeError:
        old = detector.input_size
        detector.input_size = size
        try: return detector.detect(image,max_num=0,metric='default')[0]
        finally: detector.input_size = old

def mini_crop_bgr(img,bbox):
    h,w=img.shape[:2]; x1,y1,x2,y2=map(float,bbox)
    bw,bh=x2-x1,y2-y1
    if bw<=0 or bh<=0 or min(h,w)<2: raise ValueError('Invalid bbox')
    scale=min((h-1)/bh,(w-1)/bw,2.7)
    nw,nh=bw*scale,bh*scale; cx,cy=x1+bw/2,y1+bh/2
    left,top,right,bottom=cx-nw/2,cy-nh/2,cx+nw/2,cy+nh/2
    if left<0: right-=left;left=0
    if top<0: bottom-=top;top=0
    if right>w-1: left-=right-w+1;right=w-1
    if bottom>h-1: top-=bottom-h+1;bottom=h-1
    lx,ty,rx,by=map(int,(left,top,right,bottom))
    if lx<0 or ty<0 or rx>=w or by>=h or rx<lx or by<ty: raise RuntimeError('Crop escaped image')
    patch=img[ty:by+1,lx:rx+1]
    if patch.size==0: raise RuntimeError('Empty crop')
    return cv2.resize(patch,(80,80))

cache_dir=OUTPUT_ROOT/'cache';cache_dir.mkdir(exist_ok=True)
records=[]
for idx,row in enumerate(frame_manifest.itertuples(index=False)):
    rec={'subject_id':row.subject_id,'video_id':row.video_id,'video_code':row.video_code,
         'frame_index':row.frame_index,'image_path':row.image_path,'binary_label':row.binary_label,
         'attack_type':row.attack_type,'quality':row.quality,'decoded':False,'detected':False,
         'detector_size':'','confidence':np.nan,'bbox_xyxy':'','crop_path':'','failure_reason':''}
    image=cv2.imread(row.image_path,cv2.IMREAD_COLOR)
    if image is None: rec['failure_reason']='decode_failed'
    else:
        rec['decoded']=True
        try:
            for size in ((640,640),(480,480),(320,320)):
                bboxes=detect(image,size)
                if bboxes is None or len(bboxes)==0: continue
                # No CASIA bbox annotation exists. Choose the largest face, then confidence.
                chosen=max(bboxes,key=lambda b: ((float(b[2])-float(b[0]))*(float(b[3])-float(b[1])),float(b[4])))
                crop=mini_crop_bgr(image,chosen[:4]); tensor=crop.transpose(2,0,1).astype(np.float32)/255.0
                if tensor.shape!=(3,80,80) or not np.isfinite(tensor).all(): raise RuntimeError('Invalid input crop')
                crop_path=cache_dir/f'{idx:05d}.npy';np.save(crop_path,tensor)
                rec.update(detected=True,detector_size=str(size[0]),confidence=float(chosen[4]),
                           bbox_xyxy=json.dumps([float(x) for x in chosen[:4]]),crop_path=str(crop_path))
                break
            if not rec['detected']: rec['failure_reason']='scrfd_no_face'
        except Exception as exc: rec['failure_reason']=f'{type(exc).__name__}: {exc}'
    records.append(rec)
    if (idx+1)%250==0: print('Localized',idx+1,'/',len(frame_manifest))
detect_frame=pd.DataFrame(records)
coverage=detect_frame.groupby('video_id',sort=True).agg(planned_frames=('frame_index','size'),
    decoded_frames=('decoded','sum'),detected_frames=('detected','sum')).reset_index()
coverage=videos.drop(columns='planned_frames').merge(coverage,on='video_id',validate='one_to_one')
coverage['detection_coverage']=coverage.detected_frames/coverage.planned_frames
coverage['scorable']=coverage.detected_frames>=MIN_VALID_FRAMES
save_csv(detect_frame,'casia_frame_detection.csv')
save_csv(coverage,'casia_detection_coverage.csv')
coverage_groups=[]
for columns in (['binary_label'],['attack_type'],['quality']):
    for values, group in coverage.groupby(columns,dropna=False):
        values = values if isinstance(values,tuple) else (values,)
        coverage_groups.append({'group_by':'+'.join(columns),'group':'+'.join(map(str,values)),
            'videos':len(group),'scorable_videos':int(group.scorable.sum()),
            'video_coverage':float(group.scorable.mean()),
            'planned_frames':int(group.planned_frames.sum()),
            'detected_frames':int(group.detected_frames.sum()),
            'frame_detection_coverage':float(group.detected_frames.sum()/group.planned_frames.sum())})
save_csv(pd.DataFrame(coverage_groups),'casia_coverage_by_group.csv')
if not coverage.scorable.any(): raise RuntimeError('No scorable CASIA videos; inspect detection coverage')
if set(coverage.loc[coverage.scorable,'binary_label']) != {0,1}:
    raise RuntimeError('Scorable videos lack Real or Attack class; primary PAD metrics are undefined')
print('Scorable videos:',int(coverage.scorable.sum()),'/',len(coverage))
display(coverage.groupby(['binary_label','attack_type','quality']).agg(videos=('video_id','size'),scorable=('scorable','sum'),
    planned_frames=('planned_frames','sum'),detected_frames=('detected_frames','sum')).reset_index())


Localized 250 / 2405
Localized 500 / 2405
Localized 750 / 2405
Localized 1000 / 2405
Localized 1250 / 2405
Localized 1500 / 2405
Localized 1750 / 2405
Localized 2000 / 2405
Localized 2250 / 2405
Scorable videos: 352 / 360


,binary_label,attack_type,quality,videos,scorable,planned_frames,detected_frames
0,0,real,high,30,30,218,218
1,0,real,low,30,28,172,172
2,0,real,normal,30,30,198,198
3,1,cut,high,30,28,158,158
4,1,cut,low,30,30,168,168
5,1,cut,normal,30,30,210,210
6,1,replay,high,30,30,208,208
7,1,replay,low,30,30,203,203
8,1,replay,normal,30,26,158,158
9,1,warped,high,30,30,282,282


In [12]:
# Four independent frozen ONNX models consume the identical ordered crop cache.
valid = detect_frame[detect_frame.detected].reset_index(drop=True)
assert valid.crop_path.is_unique and all(Path(p).is_file() for p in valid.crop_path)
frame_predictions=[]
for model in provenance:
    session=ort.InferenceSession(model['onnx_path'],providers=providers)
    actual=session.get_providers()[0]
    print(model['model_id'],'provider:',actual)
    logits=[]
    for start in range(0,len(valid),128):
        batch=np.stack([np.load(p) for p in valid.crop_path.iloc[start:start+128]]).astype(np.float32)
        out=session.run(['logits'],{'input':batch})[0]
        if out.shape!=(len(batch),3) or not np.isfinite(out).all(): raise RuntimeError('Invalid ONNX logits')
        logits.append(out)
    z=np.concatenate(logits)
    attack_lse=np.logaddexp(z[:,1],z[:,2])
    scores=z[:,0]-attack_lse
    part=valid[['subject_id','video_id','video_code','frame_index','image_path','binary_label','attack_type','quality','crop_path']].copy()
    part.insert(0,'model_id',model['model_id'])
    part['logit_real']=z[:,0];part['logit_physical']=z[:,1];part['logit_digital']=z[:,2]
    part['pad_score']=scores
    part['locked_threshold']=model['locked_celeba_val_logit_threshold']
    part['pred_real']=scores>=part.locked_threshold
    frame_predictions.append(part)
frame_predictions=pd.concat(frame_predictions,ignore_index=True)
keys=valid[['video_id','frame_index','image_path']].astype(str).agg('|'.join,axis=1).tolist()
assert all(part[['video_id','frame_index','image_path']].astype(str).agg('|'.join,axis=1).tolist()==keys for _,part in frame_predictions.groupby('model_id',sort=False))
save_csv(frame_predictions,'frame_predictions.csv')
video_predictions=frame_predictions.groupby(['model_id','video_id'],sort=False).agg(
    subject_id=('subject_id','first'),video_code=('video_code','first'),binary_label=('binary_label','first'),
    attack_type=('attack_type','first'),quality=('quality','first'),valid_frames=('pad_score','size'),
    video_score=('pad_score','mean'),locked_threshold=('locked_threshold','first')).reset_index()
video_predictions=video_predictions.merge(coverage[['video_id','planned_frames','decoded_frames','detected_frames','scorable']],on='video_id',validate='many_to_one')
video_predictions['pred_real']=np.where(video_predictions.scorable,video_predictions.video_score>=video_predictions.locked_threshold,np.nan)
scorable_keys=coverage.loc[coverage.scorable,'video_id'].tolist()
assert all(set(g.loc[g.scorable,'video_id'])==set(scorable_keys) for _,g in video_predictions.groupby('model_id'))
save_csv(video_predictions,'video_predictions.csv')


scratch_vanilla provider: CPUExecutionProvider
scratch_csmr provider: CPUExecutionProvider
pretrained_vanilla provider: CPUExecutionProvider
pretrained_csmr provider: CPUExecutionProvider


In [13]:
# Primary video metrics, paired deltas, and secondary diagnostics.
def pad_metrics(frame, threshold):
    y=frame.binary_label.to_numpy(dtype=int);score=frame.video_score.to_numpy(dtype=float) if 'video_score' in frame else frame.pad_score.to_numpy(dtype=float)
    real=y==0;attack=y==1
    if not real.any() or not attack.any(): return {'auc':np.nan,'apcer':np.nan,'bpcer':np.nan,'hter':np.nan}
    pred_real=score>=threshold
    apcer=float(pred_real[attack].mean());bpcer=float((~pred_real[real]).mean())
    return {'auc':float(roc_auc_score(real.astype(int),score)),'apcer':apcer,'bpcer':bpcer,'hter':(apcer+bpcer)/2}

primary=[]; frame_metrics=[];attack_rows=[];quality_rows=[]
for model in provenance:
    mid=model['model_id'];t=model['locked_celeba_val_logit_threshold']
    v=video_predictions[(video_predictions.model_id==mid)&video_predictions.scorable]
    f=frame_predictions[frame_predictions.model_id==mid]
    primary.append({'model_id':mid,'initialization':model['initialization_regime'],'method':model['method'],
                    'n_videos':len(v),'candidate_videos':len(coverage),'locked_threshold':t,**pad_metrics(v,t)})
    frame_metrics.append({'model_id':mid,'role':'secondary frame diagnostic','n_frames':len(f),**pad_metrics(f,t)})
    for attack, subset in v[v.binary_label==1].groupby('attack_type'):
        attack_rows.append({'model_id':mid,'attack_type':attack,'n_videos':len(subset),
                            'apcer':float((subset.video_score>=t).mean()),
                            'mean_score':float(subset.video_score.mean()),'median_score':float(subset.video_score.median())})
    for quality,subset in v.groupby('quality'):
        quality_rows.append({'model_id':mid,'quality':quality,'n_videos':len(subset),**pad_metrics(subset,t)})
primary=pd.DataFrame(primary); save_csv(primary,'casia_primary_metrics.csv')
save_csv(pd.DataFrame(frame_metrics),'casia_frame_level_metrics.csv')
save_csv(pd.DataFrame(attack_rows),'casia_attack_type_metrics.csv')
save_csv(pd.DataFrame(quality_rows),'casia_quality_metrics.csv')
deltas=[]
for regime in SOURCES:
    base=primary.set_index('model_id').loc[regime+'_vanilla'];csmr=primary.set_index('model_id').loc[regime+'_csmr']
    deltas.append({'initialization':regime,**{f'delta_{m}':float(csmr[m]-base[m]) for m in ('auc','apcer','bpcer','hter')}})
save_csv(pd.DataFrame(deltas),'casia_paired_deltas.csv')
display(primary[['initialization','method','auc','apcer','bpcer','hter','n_videos']]);display(pd.DataFrame(deltas))

# Paired subject bootstrap: duplicate an entire subject's videos when resampled.
subject_ids=np.array(sorted(coverage.subject_id.unique()))
assert len(subject_ids)==30
rng=np.random.default_rng(BOOTSTRAP_SEED)
bootstrap_rows=[]
for regime in SOURCES:
    pair={method:video_predictions[(video_predictions.model_id==regime+'_'+method)&video_predictions.scorable].copy() for method in ('vanilla','csmr')}
    assert pair['vanilla'].video_id.tolist()==pair['csmr'].video_id.tolist()
    distributions={'auc':[],'hter':[]}
    for _ in range(N_BOOTSTRAP):
        sampled=rng.choice(subject_ids,len(subject_ids),replace=True)
        subsets={method:pd.concat([df[df.subject_id==sid] for sid in sampled],ignore_index=True) for method,df in pair.items()}
        if any(set(subset.binary_label)!={0,1} for subset in subsets.values()): continue
        result={method:pad_metrics(subsets[method],float(subsets[method].locked_threshold.iloc[0])) for method in pair}
        for metric in distributions: distributions[metric].append(result['csmr'][metric]-result['vanilla'][metric])
    point=next(d for d in deltas if d['initialization']==regime)
    for metric,values in distributions.items():
        if not values: raise RuntimeError('Bootstrap produced no valid paired samples')
        a=np.asarray(values,dtype=float)
        bootstrap_rows.append({'initialization':regime,'metric':'delta_'+metric,
            'point_estimate':point['delta_'+metric],'bootstrap_mean':float(a.mean()),
            'bootstrap_std':float(a.std(ddof=1)),'ci95_low':float(np.percentile(a,2.5)),
            'ci95_high':float(np.percentile(a,97.5)),'valid_replicates':len(a),'requested_replicates':N_BOOTSTRAP,
            'unit':'paired CASIA image-copy test subjects'})
save_csv(pd.DataFrame(bootstrap_rows),'casia_bootstrap_summary.csv')

for regime in SOURCES:
    fig,ax=plt.subplots(figsize=(5,5))
    for method in ('vanilla','csmr'):
        frame=video_predictions[(video_predictions.model_id==regime+'_'+method)&video_predictions.scorable]
        fpr,tpr,_=roc_curve((frame.binary_label==0).astype(int),frame.video_score)
        ax.plot(fpr,tpr,label=f'{method} AUC {roc_auc_score((frame.binary_label==0).astype(int),frame.video_score):.3f}')
    ax.plot([0,1],[0,1],'k:',linewidth=1);ax.set(xlabel='False positive rate',ylabel='True positive rate',title=regime+' video-level ROC')
    ax.legend();fig.tight_layout();fig.savefig(OUTPUT_ROOT/f'roc_casia_video_{regime}.png',dpi=160);plt.close(fig)


,initialization,method,auc,apcer,bpcer,hter,n_videos
0,scratch,vanilla,0.645059,0.162879,0.568182,0.365530,352
1,scratch,csmr,0.701963,0.132576,0.545455,0.339015,352
2,pretrained,vanilla,0.687758,0.238636,0.465909,0.352273,352
3,pretrained,csmr,0.710744,0.340909,0.272727,0.306818,352


,initialization,delta_auc,delta_apcer,delta_bpcer,delta_hter
0,scratch,0.056904,-0.030303,-0.022727,-0.026515
1,pretrained,0.022986,0.102273,-0.193182,-0.045455


In [14]:
# Evidence-bound report and portable output ZIP.
coverage_summary={
 'candidate_videos':len(coverage),'scorable_videos':int(coverage.scorable.sum()),
 'unscorable_videos':int((~coverage.scorable).sum()),'video_coverage':float(coverage.scorable.mean()),
 'planned_frames':int(coverage.planned_frames.sum()),'decoded_frames':int(coverage.decoded_frames.sum()),
 'detected_frames':int(coverage.detected_frames.sum()),
 'frame_detection_coverage':float(coverage.detected_frames.sum()/coverage.planned_frames.sum())}
lines=['# Frozen MiniFASNetV2 evaluation — CASIA-FASD Kaggle image copy','',
 'This is the untouched external CASIA confirmation for the four CelebA-trained final 100K seed-100 models from Notebook 02 and 03. CASIA was not used for optimizer, CSMR lambda/gain, checkpoint, or threshold selection; no CASIA train data was used by this evaluation. LCC-FASD was external development, not untouched confirmation.','',
 '## Dataset and limits','',
 f"The evaluated split is all `test_img/test_img/color` in `{CASIA}`: {len(all_test)} available JPEG frames, 360 reconstructed source-video IDs, 30 subjects numbered 1–30. This is not the official subject-disjoint CASIA benchmark. The Kaggle copy has {len(shared_subjects)} subjects, {len(shared_videos)} video IDs, and {len(shared_frame_ids)} frame identifiers shared between its train/test directories. The source videos are unavailable, so metrics concern only extracted frames.",'',
 f"Coverage: {coverage_summary['scorable_videos']}/{coverage_summary['candidate_videos']} videos; {coverage_summary['detected_frames']}/{coverage_summary['planned_frames']} planned frames detected. Video score is the mean of valid frame logit scores; at least {MIN_VALID_FRAMES} detections are required. Failures were never replaced.",'',
 'No source face boxes are supplied. The detector selects the largest SCRFD box (confidence tie break); this fixed choice can pick an unintended face in a presentation attack. See detection coverage and box records.','',
 '## Frozen comparison','',
 'All four models received the identical ordered frames and cached SCRFD-500M/MiniFASNet crops. Each kept its own CelebA-Val logit threshold. CSMR was training-time only: ONNX inference has ordinary MiniFASNetV2 graphs and no FFT or spectral branch. Primary results are reconstructed video level.','',
 primary[['initialization','method','auc','apcer','bpcer','hter','n_videos']].to_markdown(index=False), '',
 'Paired within-initialization CSMR minus Vanilla differences:', '',pd.DataFrame(deltas).to_markdown(index=False),'',
 'Subject-level paired bootstrap intervals estimate target-subject sampling uncertainty, not training-seed uncertainty or statistical significance. Only one training seed was evaluated. More seeds are outside this run. Frame, attack-type, and quality results are diagnostics. No overall model ranking or SOTA claim is made.','',
 '## Reproducibility','',f'Detector SHA256: `{SCRFD_SHA256}`. See `evaluation_protocol.json`, manifests, provenance, coverage, and prediction CSV files.']
(OUTPUT_ROOT/'casia_frozen_evaluation_report.md').write_text('\n'.join(lines)+'\n')
(OUTPUT_ROOT/'coverage_summary.json').write_text(json.dumps(coverage_summary,indent=2,allow_nan=False)+'\n')
zip_path=OUTPUT_ROOT.with_suffix('.zip')
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUTPUT_ROOT.iterdir()):
        if p.is_file(): z.write(p,arcname=OUTPUT_ROOT.name+'/'+p.name)
print('Saved:',zip_path);display(FileLink(str(zip_path)))


Saved: /kaggle/working/minifasnetv2_casia_fasd_frozen_eval.zip


/kaggle/working/minifasnetv2_casia_fasd_frozen_eval.zip